# Drive-only source acquisition\n\nThis notebook is a thin Colab runner. It mounts the project Drive folder, clones only the code repository into the ephemeral Colab runtime, and calls the repository acquisition module. The ZIP and all eight CSVs are streamed/staged/promoted under Drive; no raw or curated data is written to `/content` or GitHub.\n\nBefore running, confirm the source-use terms. If the official URL is unavailable, stop, review/update `config/source_acquisition.yaml`, and use the Kaggle/manual provider only under the plan's fallback rules.

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\n\nfrom pathlib import Path\nimport os\nimport subprocess\nimport sys\n\nPROJECT_ROOT = Path('/content/drive/MyDrive/Retail DA - Customer & Commercial Intelligence')\nREPO_URL = 'https://github.com/susayold/retail-customer-commercial-intelligence.git'\nREPO_ROOT = Path('/content/retail-customer-commercial-intelligence')\nif not PROJECT_ROOT.is_dir():\n    raise FileNotFoundError(f'Drive project folder not found: {PROJECT_ROOT}')\nif not REPO_ROOT.is_dir():\n    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)], check=True)\nelse:\n    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)\nsubprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)\nos.environ['RETAIL_DRIVE_ROOT'] = str(PROJECT_ROOT)\nos.environ['RETAIL_DATA_ROOT'] = str(PROJECT_ROOT / '01_raw_source')\nos.environ['RETAIL_ARTIFACT_ROOT'] = str(PROJECT_ROOT)\nprint('Code checkout (ephemeral):', REPO_ROOT)\nprint('Drive project root:', PROJECT_ROOT)\nprint('Raw target:', os.environ['RETAIL_DATA_ROOT'])

In [ ]:
# Official acquisition: the repository module writes the archive/staging/raw files only on Drive.\nacquire = subprocess.run(\n    [sys.executable, '-m', 'src.acquire_source', '--provider', 'official', '--drive-root', str(PROJECT_ROOT)],\n    cwd=REPO_ROOT,\n    text=True,\n    check=False,\n)\nif acquire.returncode != 0:\n    raise RuntimeError('Official acquisition failed. Inspect Drive 06_source_docs/acquisition_manifest.json and acquisition_run.log before using a fallback.')\n\nverify = subprocess.run(\n    [sys.executable, '-m', 'src.verify_source_ready', '--drive-root', str(PROJECT_ROOT)],\n    cwd=REPO_ROOT,\n    text=True,\n    check=False,\n)\nif verify.returncode != 0:\n    raise RuntimeError('Drive source readiness verification failed; do not start business analysis.')\nprint('Drive source package is READY. Next cell updates the tracker; then run make schema inventory profile parquet warehouse validate quality-gate.')

In [ ]:
# Optional tracker update after the verifier is green. This cell writes metadata only, not source records.\nimport json\nfrom google.colab import auth\nauth.authenticate_user()\nimport google.auth\nfrom googleapiclient.discovery import build\n\nSHEET_ID = '16Iz47jiHM2nl5gGuhP5Py_xbNjLVLO5FaFpiL-_dR4k'\nprovenance_path = PROJECT_ROOT / '06_source_docs' / 'source_provenance.json'\nprovenance = json.loads(provenance_path.read_text(encoding='utf-8'))\nfiles = provenance['files']\ncredentials, _ = google.auth.default()\nsheets = build('sheets', 'v4', credentials=credentials)\nsheets.spreadsheets().values().batchUpdate(\n    spreadsheetId=SHEET_ID,\n    body={\n        'valueInputOption': 'USER_ENTERED',\n        'data': [\n            {'range': \"'Source Register'!E2:E9\", 'values': [['Verified'] for _ in files]},\n            {'range': \"'Source Register'!H2:K9\", 'values': [[provenance.get('direct_asset_url', ''), provenance['acquired_at'], 'manifest-' + provenance.get('manifest_version', '1.0'), item['sha256']] for item in files]},\n            {'range': \"'Plan Status'!C3\", 'values': [['Done']]},\n            {'range': \"'Plan Status'!D3\", 'values': [[f\"Drive source READY: exact 8-file set, checksums PASS. Evidence: {provenance_path}.\"]]},\n            {'range': \"'Plan Status'!E3\", 'values': [['Run make schema inventory profile parquet warehouse validate quality-gate; keep all outputs under Drive.']]},\n        ]\n    }\n).execute()\nprint('Source Register and Plan Status updated: Verified / Done.')